# LlamaFactory 微调准备流程

本 Notebook 完成 LoRA 微调前的全部数据与环境准备工作，包含以下四个步骤：

1. 克隆 LlamaFactory 训练框架仓库
2. 将原始 AdvertiseGen 数据集转换为 Alpaca 格式
3. 从 HuggingFace Hub 下载 DeepSeek-R1-Distill-Llama-8B 基座模型
4. 向 LlamaFactory 的 `dataset_info.json` 注册自定义训练集

## 一、克隆 LlamaFactory 仓库

使用 `git clone --depth 1` 浅克隆获取最新版本的 LlamaFactory 训练框架，`--depth 1` 仅拉取最新一次提交记录以节省带宽与磁盘空间。

In [ ]:
# !git clone：在 Jupyter 中执行 Shell 命令（前缀 ! 表示 Shell 调用）
# --depth 1   ：浅克隆，仅获取最新一次提交历史，大幅减少下载量
# git@github.com:hiyouga/LlamaFactory.git ：通过 SSH 协议克隆 LlamaFactory 官方仓库
# 克隆完成后会在当前目录下生成 LlamaFactory/ 子目录，包含完整的训练框架源码
!git clone --depth 1 git@github.com:hiyouga/LlamaFactory.git

## 二、数据集转换

AdvertiseGen 原始数据为每行一个 JSON 对象的 JSONL 格式（`content` + `summary` 字段），需转换为 LlamaFactory Alpaca 格式（`instruction` + `input` + `output` 三字段 JSON 数组）才能被训练框架识别。

### 2.1 将 AdvertiseGen JSONL 转换为 Alpaca JSON 格式

读取 `data/AdvertiseGen/train.json`（每行一个 JSON 对象），逐行解析后拼装为 Alpaca 样本列表，最终整体序列化写入 `data/AdvertiseGen/train_alpaca.json`。

In [1]:
import json          # 标准库：用于 JSON 的序列化与反序列化
import os             # 标准库：操作系统接口，用于切换工作目录
from pathlib import Path  # 标准库：跨平台路径操作

# 将工作目录切换到 notebook 所在目录，确保相对路径在任意 Kernel 启动位置下均有效
# __vsc_ipynb_file__ 是 VS Code / Jupyter 内核注入的当前 notebook 绝对路径字符串
os.chdir(Path(__vsc_ipynb_file__).parent)  # None：无返回值，仅改变进程工作目录

# 原始 JSONL 训练集路径（每行一个 JSON 对象）
INPUT_PATH = Path("data/AdvertiseGen/train.json")

# 转换后的 Alpaca 格式输出路径（整体为 JSON 数组）
OUTPUT_PATH = Path("data/AdvertiseGen/train_alpaca.json")

# LlamaFactory Alpaca 格式中 instruction 字段为所有样本共享的任务描述
INSTRUCTION = (
    "根据给定的商品属性标签，生成一段流畅、有吸引力的中文广告文案。"
    "属性标签格式为「类型#值*类型#值...」，多个属性用「*」分隔。"
)


def convert_advertise_gen_to_alpaca(
    input_path: Path,
    output_path: Path,
    instruction: str,
) -> int:
    """
    将 AdvertiseGen JSONL 数据集转换为 LlamaFactory 所需的 Alpaca JSON 格式。

    参数：
        input_path  (Path): 原始 JSONL 文件路径，每行包含 content 和 summary 字段。
        output_path (Path): 输出 JSON 文件路径，保存为包含所有样本的列表。
        instruction (str) : 所有样本共用的任务描述指令字符串。

    返回：
        int: 成功转换并写入的样本条数。

    输出文件格式示例：
        [
          {
            "instruction": "根据给定的商品属性标签，生成...",
            "input": "类型#裤*版型#宽松*...",
            "output": "宽松的阔腿裤这两年..."
          },
          ...
        ]
    """
    alpaca_samples = []  # list[dict]：存储转换后的所有 Alpaca 样本

    with input_path.open("r", encoding="utf-8") as f_in:  # 以 UTF-8 编码打开原始文件
        for line_no, line in enumerate(f_in, start=1):    # 逐行迭代，line_no 从 1 开始计数
            line = line.strip()                            # str：去除首尾空白字符（含换行符）
            if not line:                                   # 跳过空行，避免 json.loads 抛出异常
                continue

            try:
                raw = json.loads(line)                     # dict：将当前行解析为 Python 字典
            except json.JSONDecodeError as e:
                # 解析失败时打印警告并跳过，保证整体转换不中断
                print(f"[警告] 第 {line_no} 行 JSON 解析失败，已跳过：{e}")
                continue

            content = raw.get("content", "").strip()       # str：商品属性标签，缺失时取空字符串
            summary = raw.get("summary", "").strip()       # str：广告文案，缺失时取空字符串

            if not content or not summary:                 # 过滤掉字段缺失或为空的无效样本
                print(f"[警告] 第 {line_no} 行 content/summary 为空，已跳过")
                continue

            # 构造单条 Alpaca 样本字典
            alpaca_sample = {
                "instruction": instruction,  # str：统一的任务指令
                "input": content,            # str：商品属性标签（模型输入）
                "output": summary,           # str：广告文案（期望输出）
            }
            alpaca_samples.append(alpaca_sample)  # 将当前样本追加至结果列表

    # 确保输出目录存在（若不存在则递归创建）
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as f_out:  # 以 UTF-8 写模式打开输出文件
        json.dump(
            alpaca_samples,      # list[dict]：待序列化的样本列表
            f_out,
            ensure_ascii=False,  # 保留中文字符，不转义为 \uXXXX
            indent=2,            # int：缩进空格数，使输出文件人类可读
        )

    return len(alpaca_samples)  # int：返回成功写入的样本条数


# 执行转换
sample_count = convert_advertise_gen_to_alpaca(INPUT_PATH, OUTPUT_PATH, INSTRUCTION)
# int：sample_count 为实际写入的样本数量

print(f"转换完成！共写入 {sample_count} 条样本")
print(f"输出路径：{OUTPUT_PATH.resolve()}")  # 打印绝对路径，方便确认文件位置

转换完成！共写入 114599 条样本
输出路径：C:\Users\15513\Desktop\LLMPractice\data\AdvertiseGen\train_alpaca.json


## 三、预训练模型下载

将 DeepSeek-R1-Distill-Llama-8B 基座模型的全部文件（权重分片 `.safetensors`、配置文件 `config.json`、分词器文件等）下载到本地，供 LlamaFactory 在 `train.yaml` 中通过 `model_name_or_path` 字段直接引用。

### 3.1 从 HuggingFace Hub 下载 DeepSeek-R1-Distill-Llama-8B

调用 `huggingface_hub.snapshot_download` 拉取仓库快照，文件平铺写入 `model/DeepSeekR1DistillLlama8BForLlamaFactory/` 目录，无需手动解压或重命名，transformers 可直接加载。

In [1]:
from pathlib import Path                     # Path：跨平台路径操作
from huggingface_hub import snapshot_download  # snapshot_download：下载 Hub 仓库全量文件到本地目录

# LlamaFactory train.yaml 的 model_name_or_path 指向的本地目录
# 其中含文件：tokenizer_config.json、tokenizer.json 等、模型权重（*.safetensors）
# 需平铺在同一目录下，才能被 transformers 正确识别
MODEL_DIR = Path("model/DeepSeekR1DistillLlama8BForLlamaFactory")  # Path：模型根目录，对应 train.yaml 的 model_name_or_path

# 确保目标目录存在，否则递归创建它
MODEL_DIR.mkdir(parents=True, exist_ok=True)        # None：无返回值，副作用是创建目录

# snapshot_download 会将仓库中的所有文件（含 config.json、tokenizer 文件、权重分片）
# 直接以扁平结构写入 local_dir，等价于手动执行 git clone --depth 1
# 参数说明：
#   repo_id  : str  —— HuggingFace Hub 仓库 ID（组织/模型名）
#   local_dir: Path —— 文件保存的目标本地目录（新版 huggingface_hub 已自动以实体文件写入，无需额外参数）
downloaded_path = snapshot_download(          # str：返回实际写入文件的本地目录绝对路径
    repo_id="deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    local_dir=MODEL_DIR,                      # Path：所有文件平铺写入此目录
)

print(f"下载完成！模型文件保存在：{downloaded_path}")  # 打印实际保存路径，方便确认
print("train.yaml 中可设置：model_name_or_path:", str(MODEL_DIR.resolve()))  # 打印绝对路径供配置使用



Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

下载完成！模型文件保存在：C:\Users\15513\Desktop\LLMPractice\model\DeepSeekR1DistillLlama8BForLlamaFactory
train.yaml 中可设置：model_name_or_path: C:\Users\15513\Desktop\LLMPractice\model\DeepSeekR1DistillLlama8BForLlamaFactory


## 四、注册数据集配置

LlamaFactory 通过 `data/dataset_info.json` 统一管理所有可用数据集的元信息。训练配置文件 `train.yaml` 中的 `dataset` 字段需与此文件中的注册键名一一对应，因此自定义数据集必须先在此处注册方可被训练脚本识别。

### 4.1 向 dataset_info.json 注册 advertise_gen_train

读取 `LlamaFactory/data/dataset_info.json`，以 `advertise_gen_train` 为键写入 `file_name` 条目（相对路径指向上一步生成的 `train_alpaca.json`），随后回写文件，完成注册。

In [ ]:
import json                          # 标准库：JSON 序列化/反序列化
from pathlib import Path             # 标准库：跨平台路径操作

# dataset_info.json 的绝对路径（LlamaFactory 启动时从此文件读取所有数据集配置）
DATASET_INFO_PATH = Path("LlamaFactory/data/dataset_info.json")  # Path：配置文件路径

# train_alpaca.json 相对于 dataset_info.json 所在目录的相对路径
# dataset_info.json 位于 LlamaFactory/data/，数据文件位于 data/AdvertiseGen/
# 因此向上两级再进入 data/AdvertiseGen/
RELATIVE_FILE_PATH = "../../data/AdvertiseGen/train_alpaca.json"  # str：LlamaFactory 运行时解析的相对路径

# 新数据集在 dataset_info.json 中的注册键名（train.yaml 的 dataset 字段引用此名称）
DATASET_KEY = "advertise_gen_train"  # str：数据集注册名

# Alpaca 格式的注册条目
# LlamaFactory 默认 Alpaca 列名为 instruction / input / output，与 train_alpaca.json 一致，
# 因此无需显式声明 columns 字段
NEW_ENTRY = {                         # dict：待写入 dataset_info.json 的单条数据集配置
    "file_name": RELATIVE_FILE_PATH,  # str：数据文件相对路径（相对于 dataset_info.json 所在目录）
}


def register_dataset(
    dataset_info_path: Path,
    key: str,
    entry: dict,
) -> None:
    """
    将新数据集条目注册到 LlamaFactory 的 dataset_info.json 文件中。

    参数：
        dataset_info_path (Path): dataset_info.json 的文件路径。
        key               (str) : 数据集注册键名，对应 train.yaml 中 dataset 字段的值。
        entry             (dict): 数据集配置字典，至少包含 file_name 字段。

    返回：
        None

    副作用：
        若 key 已存在则覆盖旧配置并打印提示；否则追加新条目后回写文件。
    """
    # 读取现有 dataset_info.json 内容（dict：键为数据集名，值为配置字典）
    with dataset_info_path.open("r", encoding="utf-8") as f:
        dataset_info: dict = json.load(f)   # dict：完整的数据集注册表

    # 判断是否已存在同名数据集
    if key in dataset_info:
        print(f"[提示] '{key}' 已存在，将覆盖旧配置：{dataset_info[key]}")

    # 写入或更新条目
    dataset_info[key] = entry   # dict：将新条目插入注册表

    # 回写文件，保持格式与原文件一致（2 空格缩进，保留中文字符）
    with dataset_info_path.open("w", encoding="utf-8") as f:
        json.dump(
            dataset_info,        # dict：完整注册表
            f,
            ensure_ascii=False,  # bool：保留中文，不转义为 \uXXXX
            indent=2,            # int：缩进空格数
        )

    print(f"注册成功！数据集键名：'{key}'")
    print(f"配置内容：{entry}")
    print(f"train.yaml 中设置：dataset: {key}")


# 执行注册
register_dataset(DATASET_INFO_PATH, DATASET_KEY, NEW_ENTRY)
